# US Top 50 — Exploratory Data Analysis
Run this notebook BEFORE building the dashboard. Goal: understand the data deeply.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from src.preprocessing import load_clean, validate
from src.feature_engineering import enrich, build_song_stats, build_artist_stats

pd.set_option('display.max_columns', None)
sns.set_theme(style='darkgrid')
print('Setup complete')

## 1. Load & Validate

In [ ]:
df_raw = load_clean('../data/top50_us.csv')
val    = validate(df_raw)

print(f'Shape: {df_raw.shape}')
print(f'Date range: {df_raw.date.min()} → {df_raw.date.max()}')
print(f'Validation report: {val}')
df_raw.head()

In [ ]:
print('Dtypes:')
print(df_raw.dtypes)
print('\nNull counts:')
print(df_raw.isnull().sum())

## 2. Feature Engineering

In [ ]:
df         = enrich(df_raw)
song_stats = build_song_stats(df)
art_stats  = build_artist_stats(df)

print(f'Enriched df shape: {df.shape}')
print(f'Unique songs: {df.song_key.nunique()}')
print(f'Unique artists: {df.artist_key.nunique()}')
song_stats.head()

## 3. Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['popularity'].hist(ax=axes[0,0], bins=30, color='steelblue')
axes[0,0].set_title('Popularity Distribution')

df['position'].value_counts().sort_index().plot(kind='bar', ax=axes[0,1], color='coral')
axes[0,1].set_title('Position Distribution')

df['duration_min'].hist(ax=axes[1,0], bins=30, color='green')
axes[1,0].set_title('Duration Distribution (minutes)')

song_stats['days_on_chart'].hist(ax=axes[1,1], bins=30, color='purple')
axes[1,1].set_title('Days on Chart Distribution')

plt.tight_layout()
plt.show()

## 4. Popularity vs Rank Correlation

In [ ]:
corr = df[['popularity','position']].corr()
print('Correlation matrix:')
print(corr)

px.scatter(
    df.sample(3000, random_state=42),
    x='position', y='popularity',
    trendline='ols',
    title=f'Popularity vs Rank (r={corr.iloc[0,1]:.3f})',
    opacity=0.4
).show()

## 5. Top Artists

In [ ]:
print('Top 15 Artists by Appearances:')
print(art_stats[['artist','total_appearances','unique_songs','dominance_index']].head(15).to_string(index=False))

## 6. Explicit Content Analysis

In [ ]:
from src.analytics import explicit_vs_clean
print(explicit_vs_clean(df))

## 7. Save Cleaned Data

In [ ]:
df.to_csv('../data/top50_enriched.csv', index=False)
song_stats.to_csv('../data/song_stats.csv', index=False)
art_stats.to_csv('../data/artist_stats.csv', index=False)
print('All files saved.')